In [1]:
# !pip install ultralytics -q

In [7]:
import pandas as pd
import torch
from ultralytics import YOLO
import os
import yaml
import pandas as pd
from collections import Counter

print(torch.cuda.is_available())


True


In [ ]:
# Uncomment if running on Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# path = "/content/drive/MyDrive/pubmat-checker"


## Create YAML File

In [8]:


data = {
    "path": "./data",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",

    "names": {
        0: "NYC_Correct",
        1: "NYC_Incorrect",
        2: "BP_Correct",
        3: "BP_Incorrect",
        4: "SK_Correct",
        5: "SK_Incorrect",
        6: "YORP_Correct",
        7: "YORP_Incorrect",
    }

}
with open("./data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml file created.")


data.yaml file created.


In [13]:
import os
import yaml
from collections import Counter

data_yaml_path = "./data.yaml"

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

base_path = data_config["path"]
class_names = data_config["names"]

image_dir = os.path.join(base_path, "images", "train")
label_dir = os.path.join(base_path, "labels", "train")

# collect files
image_stems = {
    os.path.splitext(f)[0]
    for f in os.listdir(image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
}

label_stems = {
    os.path.splitext(f)[0]
    for f in os.listdir(label_dir)
    if f.lower().endswith(".txt")
}

# check mismatches
errors = []

for stem in image_stems - label_stems:
    errors.append(f"Image has no label: {stem}")

for stem in label_stems - image_stems:
    errors.append(f"Label has no image: {stem}")

# count classes
class_counts = Counter()

for stem in label_stems:
    with open(os.path.join(label_dir, stem + ".txt")) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

# print distribution
total = sum(class_counts.values())

print("\nTraining Set Class Distribution")
for class_id, count in sorted(class_counts.items()):
    name = class_names.get(class_id, f"Unknown {class_id}")
    pct = (count / total) * 100 if total else 0
    print(f"{name}: {count} ({pct:.1f}%)")

# print errors
print("\nData Check")
if errors:
    for e in errors:
        print("-", e)
else:
    print("No errors found. All images have corresponding labels and vice versa.")


Training Set Class Distribution
NYC_Correct: 596 (14.2%)
NYC_Incorrect: 517 (12.3%)
BP_Correct: 479 (11.4%)
BP_Incorrect: 629 (15.0%)
SK_Correct: 597 (14.2%)
SK_Incorrect: 396 (9.4%)
YORP_Correct: 609 (14.5%)
YORP_Incorrect: 372 (8.9%)

Data Check
No errors found. All images have corresponding labels and vice versa.


## Train YOLO Model

### Baseline

In [14]:
# Initialize pre-trained YOLO model
model = YOLO("yolov8n.pt")

model.train(
    data="./data.yaml",
    name="base_model",
    epochs=80,
    imgsz=640,
    batch=16,
    seed=42,              

    optimizer="AdamW",
    lr0=0.003,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,

    patience=15,
    device=0,
    workers=2,
    cache=True,         
    amp=True,

    degrees=0.0,          # no rotation
    translate=0.05,       # slight movement
    scale=0.8,            # small variation only
    fliplr=0.0,           # no horizontal flip 
    flipud=0.0,
    mosaic=1.0,
    mixup=0.0             # avoid mixing correct/incorrect classes
)


New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=base_model,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000236D4C33F50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [15]:
base_model = YOLO("runs/detect/base_model/weights/best.pt")


### Tuning

In [16]:
# Re-initialize before tuning
model = YOLO("yolov8n.pt")

model.tune(
    data="./data.yaml",
    iterations=50,
    imgsz=640,
    batch=16,
    epochs=30,
    patience=10,
    optimizer="AdamW",
    plots=True,
    save=True
)


Tuner: Initialized Tuner instance with 'tune_dir=C:\Users\user\Documents\DARLYN\pubmat-checker\runs\detect\tune'
Tuner:  Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/50 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
ERROR training failure for hyperparameter tuning iteration 1
Command '['c:\\python313\\python.exe', '-m', 'ultralytics.cfg.__init__', 'train', 'task=detect', 'mode=train', 'model=yolov8n.pt', 'epochs=30', 'time=None', 'patience=10', 'batch=16', 'imgsz=640', 'save=True', 'save_period=-1', 'cache=False', 'device=None', 'workers=

### Tuned Model (Best Params)

In [17]:
# Re-initialize before full tuned training run
model = YOLO("yolov8s.pt")

model.train(
    data="./data.yaml",
    name="tuned_model",
    epochs=150,
    patience=20,
    device=0,
    workers=4,
    imgsz=640,
    batch=16,
    seed=42,              # fixed seed for reproducibility

    # Best hyperparameters from tuning

    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    box=7.5,
    cls=0.5,
    cls_pw=0.0,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,           # overridden: logos are orientation-sensitive
    bgr=0.0,
    mosaic=1.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    close_mosaic=10
)


New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tuned_mode

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000237EE246820>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [18]:
tuned_model = YOLO("runs/detect/tuned_model/weights/best.pt")


### Tuned Model (Longer Training)

In [19]:
# Re-initialize before extended training run
model = YOLO("yolov8s.pt")

# NOTE: imgsz=800 increases memory and compute per batch significantly.
# If you experience OOM errors, reduce batch size or revert to imgsz=640.
model.train(
    data="./data.yaml",
    name="tuned_model_v2",
    epochs=300,
    patience=30,
    device=0,
    workers=4,
    imgsz=800,
    batch=8,              
    seed=42,              

    # Best hyperparameters from tuning
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    box=7.5,
    cls=0.5,
    cls_pw=0.0,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,           
    bgr=0.0,
    mosaic=1.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    close_mosaic=10
)


New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tuned_model

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000236D08EE820>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [20]:
tuned_model_v2 = YOLO("runs/detect/tuned_model_v2/weights/best.pt")


## Metrics

In [60]:
def get_metrics(model, ver, split):
    metrics = model.val(
        data="./data.yaml",
        plots=True,
        save=True,
        save_json=True,
        project="./results",
        name=f"{ver}_{split}",
        split=split,
        verbose=False
    )

    metrics_df = pd.DataFrame({
        "mAP50": [metrics.box.map50],
        "mAP50-95": [metrics.box.map],
        "Precision": [metrics.box.mp],
        "Recall": [metrics.box.mr]
    })

    # Per-class metrics from class_result()
    rows = []
    for i, class_name in metrics.names.items():
        p, r, ap50, ap = metrics.box.class_result(i)
        rows.append({
            "Class": class_name,
            "Precision": p,
            "Recall": r,
            "mAP50": ap50,
            "mAP50-95": ap
        })

    per_class_df = pd.DataFrame(rows)

    return metrics_df, per_class_df

### Evaluate All Models (Test Set)

In [ ]:


base_metrics_df, base_per_class_df = get_metrics(base_model, "base_model", "test")
tuned_metrics_df, tuned_per_class_df = get_metrics(tuned_model, "tuned_model", "test")
tuned_metrics_df_v2, tuned_per_class_df_v2 = get_metrics(tuned_model_v2, "tuned_model_v2", "test")


Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 954.6621.9 MB/s, size: 225.6 KB)
val: Scanning C:\Users\user\Documents\DARLYN\pubmat-checker\data\labels\test.cache... 320 images, 30 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 320/320 111.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 5.2it/s 3.8s0.1s
                   all        320        923      0.948       0.97      0.984      0.945
Speed: 1.8ms preprocess, 4.4ms inference, 0.0ms loss, 1.6ms postprocess per image
Saving C:\Users\user\Documents\DARLYN\pubmat-checker\runs\detect\results\base_model_test4\predictions.json...
Results saved to C:\Users\user\Documents\DARLYN\pubmat-checker\runs\detect\results\base_model_test4
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 89

In [55]:
print("Metrics for Models on Test Set:")
summarized_metrics = pd.concat([base_metrics_df, tuned_metrics_df, tuned_metrics_df_v2])
summarized_metrics.index = ["Base Model", "Tuned Model", "Tuned Model v2"]
summarized_metrics

Metrics for Models on Test Set:


,mAP50,mAP50-95,Precision,Recall
Base Model,0.984409,0.945058,0.947933,0.970331
Tuned Model,0.989044,0.970067,0.966370,0.977421
Tuned Model v2,0.986810,0.964807,0.973215,0.975659


In [57]:
print("\nPer-class Metrics for Tuned Model on Test Set:")
tuned_per_class_df


Per-class Metrics for Tuned Model on Test Set:


,Class,Precision,Recall,mAP50,mAP50-95
0,NYC_Correct,0.964625,0.992481,0.991414,0.990365
1,NYC_Incorrect,0.991020,0.985416,0.987411,0.975644
2,BP_Correct,0.919753,1.000000,0.974765,0.974543
3,BP_Incorrect,0.980661,0.946565,0.989605,0.970449
4,SK_Correct,0.943366,1.000000,0.991361,0.991361
5,SK_Incorrect,1.000000,0.951050,0.993432,0.936912
6,YORP_Correct,0.955386,0.991935,0.994437,0.993250
7,YORP_Incorrect,0.976152,0.951919,0.989925,0.928011


In [58]:
print("\nPer-class Metrics for Tuned Model v2 on Test Set:")
tuned_per_class_df_v2


Per-class Metrics for Tuned Model v2 on Test Set:


,Class,Precision,Recall,mAP50,mAP50-95
0,NYC_Correct,0.970627,0.993826,0.992649,0.991533
1,NYC_Incorrect,0.980961,0.973214,0.993566,0.980989
2,BP_Correct,0.937299,1.000000,0.979384,0.978017
3,BP_Incorrect,0.992036,0.950874,0.990582,0.972974
4,SK_Correct,0.959186,1.000000,0.985945,0.980466
5,SK_Incorrect,0.988083,0.968750,0.979479,0.905814
6,YORP_Correct,0.958845,1.000000,0.985789,0.983698
7,YORP_Incorrect,0.998683,0.918605,0.987087,0.924963


### Evaluate All Models (Val Set)

In [62]:

base_metrics_val_df, base_per_class_df = get_metrics(base_model, "base_model", "val")
tuned_metrics_val_df, tuned_per_class_df = get_metrics(tuned_model, "tuned_model", "val")
tuned_metrics_val_df_v2, tuned_per_class_df_v2 = get_metrics(tuned_model_v2, "tuned_model_v2", "val")

Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 1420.7284.7 MB/s, size: 262.5 KB)
val: Scanning C:\Users\user\Documents\DARLYN\pubmat-checker\data\labels\val.cache... 334 images, 27 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 334/334 127.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 4.7it/s 4.5s0.2s
                   all        334        992      0.956      0.969      0.984       0.94
Speed: 1.5ms preprocess, 4.9ms inference, 0.0ms loss, 1.9ms postprocess per image
Saving C:\Users\user\Documents\DARLYN\pubmat-checker\runs\detect\results\base_model_val4\predictions.json...
Results saved to C:\Users\user\Documents\DARLYN\pubmat-checker\runs\detect\results\base_model_val4
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 1993

In [63]:
print("Metrics for Models on Validation Set:")
summarized_metrics_val = pd.concat([base_metrics_val_df, tuned_metrics_val_df, tuned_metrics_val_df_v2])
summarized_metrics_val.index = ["Base Model", "Tuned Model", "Tuned Model v2"]
summarized_metrics_val

Metrics for Models on Validation Set:


,mAP50,mAP50-95,Precision,Recall
Base Model,0.984379,0.939770,0.956346,0.968665
Tuned Model,0.988317,0.964467,0.978304,0.979841
Tuned Model v2,0.989134,0.965483,0.978935,0.977560


In [65]:
print("Per-class Metrics for Tuned Model on Validation Set:")
tuned_per_class_df

Per-class Metrics for Tuned Model on Validation Set:


,Class,Precision,Recall,mAP50,mAP50-95
0,NYC_Correct,0.976818,0.993333,0.993733,0.989527
1,NYC_Incorrect,0.999560,0.990991,0.995000,0.978379
2,BP_Correct,0.947573,0.988438,0.987361,0.981084
3,BP_Incorrect,0.988623,0.964539,0.993202,0.967151
4,SK_Correct,0.959550,0.992857,0.987201,0.982160
5,SK_Incorrect,0.996087,0.954023,0.962706,0.914109
6,YORP_Correct,0.961621,1.000000,0.994597,0.991537
7,YORP_Incorrect,0.996598,0.954545,0.992736,0.911792


In [66]:
print("Per-class Metrics for Tuned Model v2 on Validation Set:")
tuned_per_class_df_v2

Per-class Metrics for Tuned Model v2 on Validation Set:


,Class,Precision,Recall,mAP50,mAP50-95
0,NYC_Correct,0.976785,0.993333,0.993855,0.990550
1,NYC_Incorrect,0.996608,0.981982,0.994910,0.981640
2,BP_Correct,0.954335,0.979645,0.983378,0.975183
3,BP_Incorrect,0.984588,0.971631,0.987762,0.967351
4,SK_Correct,0.978950,0.996549,0.992009,0.983672
5,SK_Incorrect,0.983132,0.965517,0.988028,0.927577
6,YORP_Correct,0.957787,1.000000,0.990255,0.988275
7,YORP_Incorrect,0.999296,0.931818,0.982877,0.909614


### Model Comparison Summary

In [70]:
train_df = summarized_metrics.copy()
train_df["Split"] = "Test"

val_df = summarized_metrics_val.copy()
val_df["Split"] = "Val"

combined = pd.concat([train_df, val_df]).reset_index().rename(columns={"index": "Model"})
combined = combined.set_index(["Model", "Split"]).round(4)

combined

,,mAP50,mAP50-95,Precision,Recall
Model,Split,,,,
Base Model,Test,0.9844,0.9451,0.9479,0.9703
Tuned Model,Test,0.9890,0.9701,0.9664,0.9774
Tuned Model v2,Test,0.9868,0.9648,0.9732,0.9757
Base Model,Val,0.9844,0.9398,0.9563,0.9687
Tuned Model,Val,0.9883,0.9645,0.9783,0.9798
Tuned Model v2,Val,0.9891,0.9655,0.9789,0.9776
